# Notebook 05 - Cox proportional hazards

The arm that gets monotonicity for free, and pays for it with an assumption.

Cox factorises the hazard into a baseline that depends only on time and a
multiplier that depends only on the covariates:

$$\lambda(x, t) = \lambda_0(t)\,\exp(x^{\top}\beta)
\qquad\Longrightarrow\qquad
\Lambda(x, t) = \Lambda_0(t)\,\exp(x^{\top}\beta)$$

$\Lambda_0$ is the Breslow estimator, a non-decreasing step function, and
$\exp(x^{\top}\beta) > 0$ always. Their product is non-decreasing in $t$ for
every borrower, so $S = e^{-\Lambda}$ is monotone non-increasing **by
construction**. There is no penalty, no collocation grid and nothing to tune:
the monotonicity audit on this arm should return exactly zero violations, and
if it does not, the implementation is wrong.

## The price

That factorisation is the proportional hazards assumption. It says the *ratio*
of two borrowers' hazards is constant over the life of the loan. A borrower who
is twice as risky at month 6 must still be exactly twice as risky at month 48.

Credit does not obviously work that way. Interest rate and grade are Lending
Club's own underwriting decision, and underwriting is sharpest about early
default; its discriminating power would be expected to decay as loans season.
If that is true in this data, PH is violated, and the violation is the honest
argument for a hazard model flexible enough to let the effect of a covariate
change shape over time.

This notebook fits the model, tests the assumption, and reports which
covariates break it.

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.common import (load_data, make_split, build_features, describe_split,
                        survival_arrays, subsample_train, HORIZONS, SEED,
                        SUBSAMPLE_N, RESULTS_DIR, IBS_GRID)
from src.evaluate import evaluate_arm, save_arm_results, plot_calibration
from src.monotonicity import audit_monotonicity, format_audit

warnings.filterwarnings("ignore", category=FutureWarning)
RESULTS_DIR.mkdir(exist_ok=True)
print("lifelines CoxPHFitter ready")

## The shared protocol

Same split and same feature matrix as every other arm.

**Training subsample.** Cox is fitted on the stratified draw of
`SUBSAMPLE_N = 300,000` training rows used by the neural arms, so those four are
mutually comparable on training size as well as on features. The test split is
untouched at 674,272 rows.

In [ ]:
df = load_data()
train_df_full, test_df = make_split(seed=SEED)
train_df = subsample_train(train_df_full, n=SUBSAMPLE_N)

print(f"full train rows      : {len(train_df_full):,}")
print(f"subsampled train rows: {len(train_df):,}  "
      f"(event rate {train_df['event'].mean():.6f})")
print(f"test rows            : {len(test_df):,}  (not subsampled)")
print()

feat = build_features(train_df, test_df)
t_tr, e_tr = survival_arrays(train_df)
t_te, e_te = survival_arrays(test_df)

print(describe_split(train_df, test_df).to_string(index=False))
print()
print("design matrix:", feat.X_train.shape, feat.X_test.shape)
print("features:", feat.names)

## Fit

One deviation from the other arms, recorded here and in the results JSON: the
fit carries a small ridge penalty, `penalizer=0.1`.

The shared design matrix one-hot encodes `home_ownership` without dropping a
level, so those columns are exactly linearly dependent and the Cox information
matrix is singular without regularisation. One of the levels (`ANY`) is also
near-constant. The regularised baselines in notebook 02 absorb this through
their own L2; Cox needs it stated. The alternative -- giving Cox a different
feature matrix from every other arm -- would break the comparison, which matters
more than an unpenalised fit.

In [ ]:
cox_df = pd.DataFrame(feat.X_train, columns=feat.names)
cox_df["time"] = t_tr
cox_df["event"] = e_tr

cph = CoxPHFitter(penalizer=0.1)
cph.fit(cox_df, duration_col="time", event_col="event", show_progress=False)

print(f"log-likelihood       : {cph.log_likelihood_:,.1f}")
print(f"concordance (train)  : {cph.concordance_index_:.4f}")
print()
print(cph.summary[["coef", "exp(coef)", "se(coef)", "z", "p"]]
      .sort_values("coef", key=abs, ascending=False).round(5).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
s = cph.summary.sort_values("coef")
ax.errorbar(s["coef"], range(len(s)),
            xerr=1.96 * s["se(coef)"], fmt="o", capsize=3)
ax.axvline(0, color="crimson", lw=1.2)
ax.set_yticks(range(len(s))); ax.set_yticklabels(s.index, fontsize=8)
ax.set_xlabel(r"$\beta$ (log hazard ratio per SD)")
ax.set_title("Cox coefficients with 95% intervals")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Testing the proportional hazards assumption

The test regresses the scaled Schoenfeld residuals of each covariate on a
transform of time. Under PH, a covariate's effect does not drift, so those
residuals carry no time trend and the statistic is small. A small p-value means
the covariate's hazard ratio **does** move over the life of the loan, which is
exactly what the model forbids.

**Why this runs on its own fit.** lifelines computes the residuals against the
exact frame a model was fitted on, so the test cannot simply be pointed at a
subsample of the main fit. It also costs about a minute per 40,000 rows, which
puts the full 300,000-row fit out of reach here. So a second Cox model is fitted
on a 60,000-row draw purely for this test, and its coefficients are printed
beside the main model's to confirm the two agree. What is being tested is
whether proportional hazards holds in this data, not a property of one
particular fit, so a smaller consistent fit answers the question.

At 60,000 rows the test still has roughly 7,700 events behind it. Note that at
any sample size like this, very small departures from proportionality reach
significance, so the **ranking** of the test statistics carries more information
than the p-values on their own.

In [ ]:
PH_N = 60_000
ph_sample = cox_df.sample(n=min(PH_N, len(cox_df)), random_state=SEED)

cph_ph = CoxPHFitter(penalizer=0.1)
cph_ph.fit(ph_sample, duration_col="time", event_col="event", show_progress=False)

agree = pd.DataFrame({"coef_main_300k": cph.params_,
                      "coef_ph_fit_60k": cph_ph.params_})
agree["abs_diff"] = (agree["coef_main_300k"] - agree["coef_ph_fit_60k"]).abs()
print(f"PH-test fit on {len(ph_sample):,} rows "
      f"({int(ph_sample['event'].sum()):,} events)")
print("coefficient agreement between the two fits:")
print(agree.round(4).to_string())
print(f"largest coefficient difference: {agree['abs_diff'].max():.4f}")
print()

ph = proportional_hazard_test(cph_ph, ph_sample, time_transform="rank")
ph_tab = ph.summary.copy()
ph_tab["violates_PH_at_0.05"] = ph_tab["p"] < 0.05
ph_tab = ph_tab.sort_values("test_statistic", ascending=False)

print("PROPORTIONAL HAZARDS TEST (scaled Schoenfeld residuals, rank transform)")
print(ph_tab.round(5).to_string())
print()
viol = ph_tab.index[ph_tab["violates_PH_at_0.05"]].tolist()
print(f"covariates violating PH at p < 0.05: {len(viol)} of {len(ph_tab)}")
for v in viol:
    print("   -", v, f"(statistic {ph_tab.loc[v, 'test_statistic']:.1f})")

ph_tab.to_csv(RESULTS_DIR / "nb05_ph_assumption_test.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
o = ph_tab.sort_values("test_statistic")
colors = ["indianred" if v else "steelblue" for v in o["violates_PH_at_0.05"]]
ax.barh(range(len(o)), o["test_statistic"], color=colors)
ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=8)
ax.set_xlabel("PH test statistic (larger = stronger evidence against PH)")
ax.set_title("Proportional hazards test by covariate\nred = violates at p < 0.05")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.show()

## Prediction

Rather than materialise a survival function per test borrower through lifelines,
the two Cox pieces are combined directly:

$$S(x, t) = \exp\!\big(-\Lambda_0(t)\,e^{x^{\top}\beta}\big)$$

`baseline_cumulative_hazard_` gives $\Lambda_0$ as a step function, which is
stepped forward to the requested times, and `predict_partial_hazard` gives
$e^{x^{\top}\beta}$. This is exact, not an approximation, and it keeps a
674,272-row test set crossed with a 60-point grid inside memory.

In [ ]:
bch = cph.baseline_cumulative_hazard_
base_t = bch.index.to_numpy(float)
base_H = bch.iloc[:, 0].to_numpy(float)


def predict_survival(X, times):
    """S(x, t) = exp(-Lambda_0(t) * exp(x'beta))."""
    times = np.atleast_1d(np.asarray(times, float))
    Xdf = pd.DataFrame(np.asarray(X, float), columns=feat.names)
    ph_mult = cph.predict_partial_hazard(Xdf).to_numpy(float).reshape(-1, 1)
    idx = np.searchsorted(base_t, times, side="right") - 1
    H0 = np.where(idx < 0, 0.0, base_H[np.clip(idx, 0, len(base_H) - 1)])
    return np.exp(-H0[None, :] * ph_mult)


def predict_cumhaz(X, times):
    times = np.atleast_1d(np.asarray(times, float))
    Xdf = pd.DataFrame(np.asarray(X, float), columns=feat.names)
    ph_mult = cph.predict_partial_hazard(Xdf).to_numpy(float).reshape(-1, 1)
    idx = np.searchsorted(base_t, times, side="right") - 1
    H0 = np.where(idx < 0, 0.0, base_H[np.clip(idx, 0, len(base_H) - 1)])
    return H0[None, :] * ph_mult


grid = np.linspace(1, 60, 300)
Sc = predict_survival(feat.X_test[:5], grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for i in range(5):
    axes[0].plot(grid, Sc[i], lw=1.5, label=f"borrower {i + 1}")
axes[0].set_xlabel("Time (months)"); axes[0].set_ylabel("S(t)")
axes[0].set_title("Cox PH - survival curves")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].plot(base_t, base_H, lw=1.8)
axes[1].set_xlim(0, 60)
axes[1].set_xlabel("Time (months)"); axes[1].set_ylabel(r"$\Lambda_0(t)$")
axes[1].set_title("Breslow baseline cumulative hazard (non-decreasing)")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Monotonicity audit

Run exactly as for every other arm. The expected answer is zero, and getting a
zero here is a check on the audit as much as on the model: it confirms the audit
is not reporting violations that come from the measurement itself.

In [ ]:
mono = audit_monotonicity(predict_cumhaz, feat.X_test, name="Cox PH",
                          n_borrowers=1000)
print(format_audit(pd.DataFrame([mono])).T.to_string(header=False))
print()
if mono["pct_points_violating"] == 0.0:
    print("Zero violations, as the factorisation guarantees.")
else:
    print("NON-ZERO violations from a Cox model: the implementation is wrong.")

## Evaluation

In [ ]:
result = evaluate_arm(predict_survival, feat.X_test, test_df, train_df,
                     name="Cox PH")
print(result)
print()
print("notes:", result.notes)

save_arm_results(result, mono, extra={
    "penalizer": 0.1,
    "deviation": ("ridge penalizer=0.1; the shared one-hot encoding of "
                  "home_ownership is exactly collinear and makes the Cox "
                  "information matrix singular without it"),
    "subsample_n": SUBSAMPLE_N,
    "n_train_used": int(len(train_df)),
    "ph_violations": viol,
    "ph_n_covariates": int(len(ph_tab)),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
plot_calibration(result, ax=axes[0])
axes[1].plot(result.brier["month"], result.brier["brier"], lw=1.8)
axes[1].set_xlabel("Time (months)"); axes[1].set_ylabel("IPCW Brier score")
axes[1].set_title("Brier score over time - Cox PH"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

Cox delivers a structurally valid term structure at no cost in machinery: no
penalty weight, no collocation points, no architecture trick. Monotonicity falls
out of the factorisation.

What the factorisation costs is flexibility. Every covariate is forced to act on
the hazard by the same multiplier for the entire life of the loan. The
assumption test above says which covariates that misrepresents. Where PH is
violated, a model that lets a covariate's effect change shape over time has a
real reason to exist -- and the monotone-architecture network in notebook 06 is
that model, with monotonicity still structural rather than penalised.